In [48]:
using PEPSKit, TensorKit

### Model Parameters ###
L = 3 #Length/width of unit cell
n_0 = round(Int, ((L-1)/2))+1 #Index of the point at the center of the lattice (1-based indexing).
m2 = 1.0 #Bare mass (squared)
m0 = 1 #Basis frequency
l = 0.1 #phi^4 coupling strength
Dim = 4 #Truncated local Hilbert space dimension
d = 2 #Number of spatial dimensions
a = 1.0 #Lattice spacing

### iPEPS Dimensions ###
Dbond = 4
χ = 32

### phi4 Hamiltonian ###
include("phi4_Hamiltonian.jl")
H, φ, φ2, φ4, Π, Π2 = phi4_model(L, m2, m0, l, Dim, d, a)

using JLD2
peps_1 = load_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")
env_1 = load_object("Z:/Energy Correlator 2D/VacStates/env,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")

CTMRGEnv{TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}, TensorMap{ComplexF64, ComplexSpace, 3, 1, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}(ComplexF64[-0.623163333771906 + 0.7737351249677901im, -0.00025818382682449016 + 0.00044942196133735746im, 0.000646321805614567 + 0.0005500952473513024im, 1.6418182752803692e-5 - 0.00019576094333609238im, -0.00022909571817270603 - 7.464339651530275e-5im, 5.197683439961586e-5 - 6.453573289717375e-5im, -1.5390293447782797e-5 + 1.9108971911027713e-5im, -3.912254518535658e-6 - 8.016484650877985e-6im, 1.1058251371507338e-6 - 1.3730202998789734e-6im, -1.8672355191327935e-6 + 2.3184062253701218e-6im  …  9.806455023660566e-7 - 3.144864064253459e-6im, 1.528387054677368e-6 + 1.8669307315563305e-6im, -3.3683367556993913e-6 + 1.0802031814256822e-5im, 6.876999162653495e-7 - 9.239917754952848e-7im, 6.606832986924269e-6 + 1.2918796868020505e-6im, 

In [49]:
#Convert the 1x1 vacuum tensor to LxL
A1 = peps_1.A[1,1] #1x1 vacuum tensor
AL = fill(A1, (L,L)) #Vacuum tensor copied over an LxL unit cell
peps_L = InfinitePEPS(AL)

InfinitePEPS{TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.017091795113360782 - 0.0003637506565163708im, 0.06206205162834553 - 0.020037750532392765im, -0.05135503331589843 - 0.042377823534891765im, -0.0027750695342006826 + 0.00332278219362903im, -0.007791740479739653 + 0.02683337573260661im, -0.014322538475952657 + 0.013016702046104762im, 0.012338710259747901 - 0.0029642706208901646im, -0.002733743982912049 + 0.0025593190149126735im, -0.016967469039629292 - 0.04058704458550155im, 0.005015626019711064 + 0.003694907594755558im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], ℂ^4 ← (ℂ^4 ⊗ ℂ^4 ⊗ (ℂ^4)' ⊗ (ℂ^4)')) TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.017091795113360782 - 0.0003637506565163708im, 0.06206205162834553 

In [50]:
#Convert the 1x1 environment to LxL
env_L = CTMRGEnv(randn, ComplexF64, peps_L, ℂ^24); #Generate structure of LxL environment

#Replace corner and edge tensors of LxL environment with the 1x1 corner and edge tensors
for r in 1:L, c in 1:L
    for dir in 1:4
        setcorner!(env_L, corner(env_1, dir, 1, 1), dir, r, c)
    end

    for dir in 1:4
        setedge!(env_L, edge(env_1, dir, 1, 1), dir, r, c)
    end
end

In [51]:
### Save PEPS and CTMRG environment ###
save_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", peps_L)
save_object("Z:/Energy Correlator 2D/VacStates/env,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", env_L)